# 第 11 章实验：IoU 与 NMS 阈值
使用少量连续坐标 xyxy 框观察贪心抑制。作业仍需独立完成向量化和类别感知版本。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

iou_threshold = 0.45
boxes = np.array([[1, 1, 6, 6], [1.5, 1.4, 6.2, 6.1], [5, 2, 9, 6], [1, 7, 4, 9]], dtype=float)
scores = np.array([0.95, 0.82, 0.75, 0.60])
print('boxes:', boxes.shape, 'scores:', scores.shape)

In [ ]:
def scalar_iou(a, b):
    left_top = np.maximum(a[:2], b[:2])
    right_bottom = np.minimum(a[2:], b[2:])
    wh = np.maximum(right_bottom - left_top, 0.0)
    intersection = wh[0] * wh[1]
    area_a = np.prod(np.maximum(a[2:] - a[:2], 0.0))
    area_b = np.prod(np.maximum(b[2:] - b[:2], 0.0))
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

matrix = np.array([[scalar_iou(a, b) for b in boxes] for a in boxes])
print(np.round(matrix, 3))

In [ ]:
def demo_nms(boxes, scores, threshold):
    remaining = list(np.argsort(scores)[::-1])
    keep = []
    while remaining:
        current = remaining.pop(0)
        keep.append(current)
        remaining = [idx for idx in remaining if scalar_iou(boxes[current], boxes[idx]) <= threshold]
    return keep

keep = demo_nms(boxes, scores, iou_threshold)
print(f'threshold={iou_threshold}, kept indices={keep}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, indices, title in [(axes[0], range(len(boxes)), 'all predictions'), (axes[1], keep, 'after NMS')]:
    for idx in indices:
        x1, y1, x2, y2 = boxes[idx]
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=2))
        ax.text(x1, y1, f'{idx}:{scores[idx]:.2f}')
    ax.set(xlim=(0, 10), ylim=(10, 0), aspect='equal', title=title)
plt.tight_layout(); plt.show()

## 破坏实验：阈值两端
阈值过低容易误删相邻目标；阈值过高会保留重复框。下面固定候选，只改变阈值。

In [ ]:
for threshold in (0.1, 0.45, 0.9):
    print(f'threshold={threshold:.2f} -> keep={demo_nms(boxes, scores, threshold)}')
print('边界接触框的 IoU:', scalar_iou(np.array([0,0,1,1.]), np.array([1,0,2,1.])))